# Room point clouds with COLMAP (Google Colab)

This notebook turns the extracted JPG frames for your **before** and **after** videos into two dense point clouds: `before_fused.ply` and `after_fused.ply`. It only runs COLMAP reconstruction; it does not create NeRF files, depth visualizations, or meshes.

## Before you start

1. In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another GPU), then run the cells from top to bottom.
2. Allow the notebook to install/build COLMAP. The first setup takes a while; it only lasts for this Colab runtime.
3. Supply the extracted JPG frames as either two Drive folders or two ZIP files from your computer. The videos are processed separately.

The source frames and generated point clouds are uploaded to/processed by Google Colab. Avoid uploading images you do not want on that service.

In [ ]:
# Install build dependencies and a CUDA-enabled COLMAP into this Colab runtime.
# Run once per new runtime; it may take 10-30 minutes.
import os, shutil, subprocess, textwrap

def run(command):
    print('+', ' '.join(map(str, command)))
    subprocess.run(command, check=True)

if shutil.which('nvidia-smi') is None:
    raise RuntimeError('No GPU runtime detected. Choose Runtime → Change runtime type → GPU, reconnect, and rerun.')
subprocess.run(['nvidia-smi'], check=True)
if shutil.which('nvcc') is None:
    raise RuntimeError('CUDA compiler nvcc is missing from this runtime; COLMAP dense reconstruction needs a CUDA-enabled runtime.')

# COLMAP uses CMake FetchContent features introduced in 3.25.
run(['python', '-m', 'pip', 'install', '-q', 'cmake>=3.25,<4'])

packages = [
    'build-essential', 'git', 'ninja-build', 'ccache',
    'libboost-program-options-dev', 'libboost-graph-dev', 'libboost-system-dev',
    'libeigen3-dev', 'libopenimageio-dev', 'openimageio-tools',
    'libmetis-dev', 'libgoogle-glog-dev', 'libgflags-dev',
    'libgtest-dev', 'libgmock-dev', 'libsqlite3-dev', 'libglew-dev',
    'libfreeimage-dev', 'libflann-dev', 'libceres-dev', 'libsuitesparse-dev',
    'libcurl4-openssl-dev', 'libssl-dev', 'libopenexr-dev',
]
run(['apt-get', 'update', '-qq'])
run(['apt-get', 'install', '-y', '-qq', *packages])

# Use the CUDA compiler provided by the selected Colab GPU runtime.
assert shutil.which('nvcc'), 'nvcc disappeared after dependency setup'
source = '/content/colmap-src'
build = '/content/colmap-build'
install = '/content/colmap-install'
shutil.rmtree(source, ignore_errors=True)
shutil.rmtree(build, ignore_errors=True)
shutil.rmtree(install, ignore_errors=True)
run(['git', 'clone', '--depth', '1', 'https://github.com/colmap/colmap.git', source])

# Build Abseil and Ceres versions compatible with the COLMAP source.
absl_source = '/content/abseil-cpp'
absl_build = '/content/abseil-build'
shutil.rmtree(absl_source, ignore_errors=True)
shutil.rmtree(absl_build, ignore_errors=True)
run(['git', 'clone', '--depth', '1', '--branch', '20230802.0', 'https://github.com/abseil/abseil-cpp.git', absl_source])
run(['cmake', '-S', absl_source, '-B', absl_build, '-G', 'Ninja', '-DCMAKE_BUILD_TYPE=Release', '-DCMAKE_INSTALL_PREFIX=/usr/local', '-DCMAKE_POSITION_INDEPENDENT_CODE=ON'])
run(['ninja', '-C', absl_build, '-j2'])
run(['ninja', '-C', absl_build, 'install'])

ceres_source = '/content/ceres-solver'
ceres_build = '/content/ceres-build'
shutil.rmtree(ceres_source, ignore_errors=True)
shutil.rmtree(ceres_build, ignore_errors=True)
run(['git', 'clone', '--depth', '1', '--branch', '2.1.0', 'https://github.com/ceres-solver/ceres-solver.git', ceres_source])
run(['cmake', '-S', ceres_source, '-B', ceres_build, '-G', 'Ninja', '-DCMAKE_BUILD_TYPE=Release', '-DCMAKE_INSTALL_PREFIX=/usr/local', '-DBUILD_TESTING=OFF', '-DBUILD_EXAMPLES=OFF'])
run(['ninja', '-C', ceres_build, '-j2'])
run(['ninja', '-C', ceres_build, 'install'])

# Build the command-line COLMAP tools with CUDA; no GUI is needed here.
cmake = shutil.which('cmake')
run([cmake, '-S', source, '-B', build, '-G', 'Ninja',
     '-DCMAKE_BUILD_TYPE=Release', f'-DCMAKE_INSTALL_PREFIX={install}',
     '-DCUDA_ENABLED=ON', '-DGUI_ENABLED=OFF',
     '-DCMAKE_CUDA_ARCHITECTURES=native',
     '-DAbsl_DIR=/usr/local/lib/cmake/absl',
     '-DCeres_DIR=/usr/local/lib/cmake/Ceres'])
run(['ninja', '-C', build, '-j2'])
run(['ninja', '-C', build, 'install'])

os.environ['PATH'] = install + '/bin:' + os.environ['PATH']
os.environ['LD_LIBRARY_PATH'] = install + '/lib:' + install + '/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')
print(subprocess.check_output(['colmap', '--help'], text=True).splitlines()[0])
print('CUDA COLMAP is ready.')

## Choose how to provide the frames

Set `INPUT_METHOD` to `drive` or `upload`, then run the next cell.

**Drive:** Put the JPGs in `MyDrive/room_change/before/` and `MyDrive/room_change/after/`.

**Upload from this PC:** Make two ZIP files named exactly `before.zip` and `after.zip`, each containing the JPG frames for that video. In the project, these are the JPGs inside `output/before/` and `output/after/`; do not include the COLMAP `sparse`, `dense`, or database files. The upload prompt lets you select both ZIPs.

In [ ]:
from pathlib import Path
import shutil, zipfile

INPUT_METHOD = 'upload'  #@param ['upload', 'drive']

if INPUT_METHOD == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/room_change')
    BEFORE_IMAGES = DRIVE_ROOT / 'before'
    AFTER_IMAGES = DRIVE_ROOT / 'after'
else:
    from google.colab import files
    uploaded = files.upload()  # Choose before.zip and after.zip from your PC
    required = {'before.zip', 'after.zip'}
    missing = required - set(uploaded)
    if missing:
        raise ValueError(f'Upload both files with these exact names: {sorted(missing)}')
    INPUT_ROOT = Path('/content/room_change_inputs')
    shutil.rmtree(INPUT_ROOT, ignore_errors=True)
    for scene in ('before', 'after'):
        destination = INPUT_ROOT / scene
        destination.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(scene + '.zip') as archive:
            root = destination.resolve()
            for item in archive.infolist():
                target = (destination / item.filename).resolve()
                if target != root and root not in target.parents:
                    raise ValueError(f'Unsafe path in {scene}.zip: {item.filename}')
            archive.extractall(destination)
    BEFORE_IMAGES = INPUT_ROOT / 'before'
    AFTER_IMAGES = INPUT_ROOT / 'after'

IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png'}
for label, folder in [('before', BEFORE_IMAGES), ('after', AFTER_IMAGES)]:
    images = [p for p in folder.rglob('*') if p.suffix.lower() in IMAGE_SUFFIXES]
    if len(images) < 2:
        raise ValueError(f'{label}: expected at least 2 JPG/PNG frames in {folder}, found {len(images)}')
    print(f'{label}: {len(images)} images from {folder}')

## Reconstruct before and after

This runs feature extraction, matching, sparse reconstruction, then dense stereo and fusion. It saves only the final dense point clouds in the results folder. If a scene fails sparse reconstruction, COLMAP may not have enough overlapping, sharp frames.

In [ ]:
from pathlib import Path
import shutil, subprocess

RESULTS = Path('/content/room_change_results')
shutil.rmtree(RESULTS, ignore_errors=True)
RESULTS.mkdir(parents=True)

def colmap_run(args):
    command = ['colmap', *map(str, args)]
    print('\n$', ' '.join(command))
    subprocess.run(command, check=True)

def reconstruct_scene(label, image_dir):
    project = RESULTS / label
    images = project / 'images'
    database = project / 'database.db'
    sparse = project / 'sparse'
    dense = project / 'dense'
    shutil.copytree(image_dir, images)
    sparse.mkdir()

    colmap_run(['feature_extractor', '--database_path', database, '--image_path', images,
                '--ImageReader.single_camera', '1', '--FeatureExtraction.use_gpu', '1'])
    colmap_run(['exhaustive_matcher', '--database_path', database, '--FeatureMatching.use_gpu', '1'])
    colmap_run(['mapper', '--database_path', database, '--image_path', images, '--output_path', sparse])
    model = sparse / '0'
    if not model.is_dir():
        raise RuntimeError(f'{label}: COLMAP did not create sparse/0; inspect image overlap and COLMAP output above.')

    colmap_run(['image_undistorter', '--image_path', images, '--input_path', model,
                '--output_path', dense, '--output_type', 'COLMAP', '--max_image_size', '2000'])
    colmap_run(['patch_match_stereo', '--workspace_path', dense, '--workspace_format', 'COLMAP',
                '--PatchMatchStereo.gpu_index', '0', '--PatchMatchStereo.geom_consistency', 'true'])
    ply = project / 'fused.ply'
    colmap_run(['stereo_fusion', '--workspace_path', dense, '--workspace_format', 'COLMAP',
                '--input_type', 'geometric', '--output_path', ply])
    if not ply.is_file() or ply.stat().st_size == 0:
        raise RuntimeError(f'{label}: COLMAP did not produce a non-empty fused point cloud.')
    print(f'{label} point cloud: {ply} ({ply.stat().st_size / 1024 / 1024:.1f} MB)')
    return ply

before_ply = reconstruct_scene('before', BEFORE_IMAGES)
after_ply = reconstruct_scene('after', AFTER_IMAGES)

## Download the point clouds

Download the ZIP and extract `before_fused.ply` and `after_fused.ply` on your PC. These are in separate COLMAP coordinate frames; they must be registered/aligned before a geometric comparison.

In [ ]:
from google.colab import files
import zipfile

archive_path = '/content/room_point_clouds.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(before_ply, 'before_fused.ply')
    archive.write(after_ply, 'after_fused.ply')
files.download(archive_path)